# Refresh silver_curated SQL analytics endpoint (in gold workspace)

Fabric Lakehouse SQL analytics endpoints do NOT auto-discover OneLake shortcut tables
or schema changes from `overwriteSchema=True` writes. Without an explicit refresh, the
gold warehouse sprocs that do `SELECT * FROM contoso_retail_silver_curated.dbo.<t>` see
stale schema (or no table at all) and fail with `Invalid object name` / `Invalid column`.

This notebook lives in the silver workspace (engineering artifacts stay in silver) and is
invoked as the first activity in `pl_gold_initial_load` / `pl_gold_incremental_load` via
cross-workspace TridentNotebook reference. It hits the Fabric REST API and synchronously
syncs all shortcut tables of the gold-workspace `contoso_retail_silver_curated` lakehouse.
Throws if any per-table sync fails.

In [ ]:
# Parameters injected by deploy.ps1
gold_workspace_id = ""
silver_curated_sep_id = ""

In [ ]:
import json, requests

if not gold_workspace_id or not silver_curated_sep_id:
    raise ValueError('gold_workspace_id and silver_curated_sep_id must be set')

token = notebookutils.credentials.getToken('https://api.fabric.microsoft.com')
url = f'https://api.fabric.microsoft.com/v1/workspaces/{gold_workspace_id}/sqlEndpoints/{silver_curated_sep_id}/refreshMetadata?preview=true'
r = requests.post(url, headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}, json={})
r.raise_for_status()
results = r.json()
print(f'Refreshed {len(results)} tables')
failures = [x for x in results if x.get('status') not in ('Success', 'NotRun')]
for x in results:
    print(f"  {x.get('status'):<10} {x.get('tableName')}")
if failures:
    raise RuntimeError(f'silver_curated SEP refresh failed for: {[f["tableName"] for f in failures]}')
notebookutils.notebook.exit('OK')